# CUDA RFF Ridge Fit-Core

Colab-ready notebook migrated from `src/us-100years/thesis_gpu.py`.


## 1) Colab Setup

If you open this notebook in Google Colab, run this cell first. Update `REPO_URL` for your fork/private repo if needed.


In [4]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/<your-user>/virtue-of-complexity-in-return-prediction.git')

if IN_COLAB:
    %pip -q install uv
    if not Path('/content/virtue-of-complexity-in-return-prediction').exists():
        !git clone "$REPO_URL" /content/virtue-of-complexity-in-return-prediction
    %cd /content/virtue-of-complexity-in-return-prediction
    !uv pip install -e ".[dev]"
else:
    print('Running outside Colab; ensure current working directory is project root.')


/content/virtue-of-complexity-in-return-prediction
Using Python 3.12.12 environment at: /usr
Resolved 158 packages in 377ms
Prepared 1 package in 401ms
Uninstalled 1 package in 0.49ms
Installed 1 package in 0.90ms
 ~ return-prediction==0.1.0 (from file:///content/virtue-of-complexity-in-return-prediction)


## 2) CUDA Environment Check


In [5]:
!uv run python scripts/check_cuda_env.py


python_version=3.12.12
platform=Linux-6.6.105+-x86_64-with-glibc2.35
torch_version=2.10.0+cu128
torch_cuda_compiled=12.8
torch_cuda_available=True
cuda_device_count=1
cuda_device_0=name:Tesla T4,cc:7.5,total_mem_gb:14.56
status=cuda_ready


## 3) Migrated CUDA Implementation (`thesis_gpu.py`)

This cell contains the migrated implementation and exposes:
- `prepare_dataset()`
- `run_fit_core_gpu(...)`
- `run_fit_core_cpu_reference(...)`


In [9]:
import torch
import importlib.util
from pathlib import Path

MODULE_PATH = Path('src/us-100years/thesis_gpu.py').resolve()
spec = importlib.util.spec_from_file_location('thesis_gpu', MODULE_PATH)
thesis_gpu = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(thesis_gpu)

GPUFitConfig = thesis_gpu.GPUFitConfig
prepare_dataset = thesis_gpu.prepare_dataset
resolve_device = thesis_gpu.resolve_device
run_fit_core_gpu = thesis_gpu.run_fit_core_gpu
run_fit_core_cpu_reference = thesis_gpu.run_fit_core_cpu_reference
print_metrics = thesis_gpu.print_metrics
oos_metrics_to_frame = thesis_gpu.oos_metrics_to_frame

TRAINING_WINDOWS = thesis_gpu.TRAINING_WINDOWS
RIDGE_ALPHAS = thesis_gpu.RIDGE_ALPHAS
GAMMA = thesis_gpu.GAMMA
RANDOM_SEED = thesis_gpu.RANDOM_SEED


## 4) Configure Run


In [7]:
# Adjust for smoke/full runs
MAX_FEATURES = 24          # smoke: 24, full: 12000
SOLVER_POLICY = 'auto'     # auto | primal | dual
CHUNK_SIZE_WINDOWS = 128
CHUNK_SIZE_FEATURES = None
PROFILE = True
DETERMINISTIC = False


## 5) Execute Fit-Core on CUDA


In [10]:
x_lagged, y_aligned, _dates = prepare_dataset()

device = resolve_device('cuda')
config = GPUFitConfig(
    windows=TRAINING_WINDOWS,
    max_features=MAX_FEATURES,
    gamma=GAMMA,
    alphas=RIDGE_ALPHAS,
    seed=RANDOM_SEED,
    dtype=torch.float32,
    device=device,
    solver_policy=SOLVER_POLICY,
    fit_only=True,
    chunk_size_windows=CHUNK_SIZE_WINDOWS,
    chunk_size_features=CHUNK_SIZE_FEATURES,
    profile=PROFILE,
    deterministic=DETERMINISTIC,
    compute_oos_metrics=False,
)

fit_metrics, _ = run_fit_core_gpu(x_lagged, y_aligned, config)
print_metrics(fit_metrics, profile=PROFILE)


NameError: name '__file__' is not defined

## 6) Compute And Persist OOS Metrics


In [ ]:
metrics_config = GPUFitConfig(
    windows=TRAINING_WINDOWS,
    max_features=MAX_FEATURES,
    gamma=GAMMA,
    alphas=RIDGE_ALPHAS,
    seed=RANDOM_SEED,
    dtype=torch.float32,
    device=device,
    solver_policy=SOLVER_POLICY,
    fit_only=True,
    chunk_size_windows=CHUNK_SIZE_WINDOWS,
    chunk_size_features=CHUNK_SIZE_FEATURES,
    profile=PROFILE,
    deterministic=DETERMINISTIC,
    compute_oos_metrics=True,
    metrics_output_path='artifacts/oos_config_metrics.csv',
    benchmark_mode='campbell_thompson',
)

fit_metrics, oos_metrics = run_fit_core_gpu(x_lagged, y_aligned, metrics_config)
print_metrics(fit_metrics, profile=metrics_config.profile)
metrics_df = oos_metrics_to_frame(oos_metrics)
metrics_df.to_csv(metrics_config.metrics_output_path, index=False)
print(f"Saved {len(metrics_df)} rows to {metrics_config.metrics_output_path}")
metrics_df.head()


## 7) Rank Configurations


In [ ]:
top_r2 = metrics_df.sort_values('r2_oos_ct', ascending=False).head(10)
bottom_r2 = metrics_df.sort_values('r2_oos_ct', ascending=True).head(10)
top_sharpe = metrics_df.sort_values('timing_sharpe_annualized', ascending=False).head(10)
bottom_sharpe = metrics_df.sort_values('timing_sharpe_annualized', ascending=True).head(10)

print('Top 10 by R2_OOS (Campbell-Thompson)')
display(top_r2[['window', 'n_features', 'alpha', 'solver_used', 'r2_oos_ct']])
print('Bottom 10 by R2_OOS (Campbell-Thompson)')
display(bottom_r2[['window', 'n_features', 'alpha', 'solver_used', 'r2_oos_ct']])
print('Top 10 by Timing Sharpe (annualized)')
display(top_sharpe[['window', 'n_features', 'alpha', 'solver_used', 'timing_sharpe_annualized']])
print('Bottom 10 by Timing Sharpe (annualized)')
display(bottom_sharpe[['window', 'n_features', 'alpha', 'solver_used', 'timing_sharpe_annualized']])
